# Smart MCQ Solver — LSTM Baseline with Weights & Biases Tracking

This notebook trains a BiLSTM classifier to score each answer option of a multiple-choice question, with full experiment tracking (metrics, curves, artifacts) logged to W&B.

**Before running on Kaggle:** add your W&B API key via Kaggle Secrets, or call `wandb.login(key="YOUR_KEY")` in the first code cell.

In [1]:
!pip install wandb -q


In [2]:
import wandb
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2004341 (23f2004341-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Imports

In [3]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import wandb


## Config

In [4]:
config = {
    "embed_dim": 256,
    "hidden_dim": 256,
    "num_layers": 2,
    "dropout": 0.4,
    "bidirectional": True,
    "max_len": 128,
    "prompt_trunc": 64,
    "option_trunc": 64,
    "batch_size": 64,
    "lr": 1e-3,
    "num_epochs": 10,
    "pos_weight": 4.0,
    "val_size": 0.10,
    "seed": 42,
}


## W&B Init

In [5]:
# wandb.login()  # uncomment and set WANDB_API_KEY if not already logged in

run = wandb.init(
    project="smart-mcq-solver",
    name="lstm-baseline",
    job_type="train",
    config=config,
    save_code=True,
)
cfg = wandb.config


wandb: setting up run nsrtdk6p
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260730_213203-nsrtdk6p
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lstm-baseline
wandb: ⭐️ View project at https://wandb.ai/23f2004341-indian-institute-of-technology-madras/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/23f2004341-indian-institute-of-technology-madras/smart-mcq-solver/runs/nsrtdk6p


## Device

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
wandb.config.update({"device": str(device)})


Device: cuda


## Load Data

In [7]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)
test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print("Train:", train.shape)
print("Test :", test.shape)

wandb.log({
    "data/train_rows": train.shape[0],
    "data/test_rows": test.shape[0],
})

# Log class balance of the correct-answer letter, and a small sample table
answer_counts = train["answer"].value_counts().sort_index()
wandb.log({
    "data/answer_distribution": wandb.plot.bar(
        wandb.Table(
            data=[[k, v] for k, v in answer_counts.items()],
            columns=["answer", "count"]
        ),
        "answer", "count",
        title="Correct-answer letter distribution (train)"
    )
})

sample_table = wandb.Table(dataframe=train.head(20))
wandb.log({"data/train_sample": sample_table})


Train: (2000, 8)
Test : (500, 7)


## Vocabulary

In [8]:
class Vocabulary:

    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.word_count = {}

    def build(self, texts):
        for text in texts:
            words = str(text).lower().split()
            for word in words:
                self.word_count[word] = self.word_count.get(word, 0) + 1

        for word, count in self.word_count.items():
            if count >= 2:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, text, max_len=128):
        words = str(text).lower().split()
        ids = [self.word2idx.get(w, 1) for w in words]

        if len(ids) < max_len:
            ids += [0] * (max_len - len(ids))
        else:
            ids = ids[:max_len]

        return ids


## Build Vocab

In [9]:
all_texts = []

for _, row in train.iterrows():
    all_texts.extend([row["prompt"], row["A"], row["B"], row["C"], row["D"], row["E"]])

for _, row in test.iterrows():
    all_texts.extend([row["prompt"], row["A"], row["B"], row["C"], row["D"], row["E"]])

vocab = Vocabulary()
vocab.build(all_texts)

print("Vocabulary Size:", len(vocab.word2idx))
wandb.config.update({"vocab_size": len(vocab.word2idx)})

# Log word-frequency histogram (top 30 words, excluding specials)
freq_items = sorted(vocab.word_count.items(), key=lambda x: x[1], reverse=True)[:30]
wandb.log({
    "data/top_words": wandb.plot.bar(
        wandb.Table(data=[[w, c] for w, c in freq_items], columns=["word", "count"]),
        "word", "count",
        title="Top 30 most frequent tokens"
    )
})


Vocabulary Size: 3816


## Dataset

In [10]:
class MCQDataset(Dataset):

    def __init__(self, df, vocab, max_len=128):
        self.samples = []

        for _, row in df.iterrows():
            prompt_ids = vocab.encode(row["prompt"], max_len)

            for option in ["A", "B", "C", "D", "E"]:
                option_ids = vocab.encode(row[option], max_len)

                combined = prompt_ids[:cfg.prompt_trunc] + option_ids[:cfg.option_trunc]

                label = 0
                if "answer" in df.columns:
                    label = 1 if row["answer"] == option else 0

                self.samples.append(
                    (torch.tensor(combined, dtype=torch.long), label)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


## Train/Val Split

In [11]:
train_df, val_df = train_test_split(
    train,
    test_size=cfg.val_size,
    random_state=cfg.seed,
    stratify=train["answer"]
)

train_dataset = MCQDataset(train_df, vocab, cfg.max_len)
val_dataset = MCQDataset(val_df, vocab, cfg.max_len)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)

wandb.log({
    "data/train_examples": len(train_dataset),
    "data/val_examples": len(val_dataset),
})


## Model

In [12]:
class LSTMClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=cfg.dropout,
            bidirectional=cfg.bidirectional
        )

        self.dropout = nn.Dropout(cfg.dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)
        output, _ = self.lstm(x)
        output = output[:, -1, :]
        output = self.dropout(output)
        logits = self.fc(output)
        return logits.squeeze(-1)


## Train

In [13]:
model = LSTMClassifier(len(vocab.word2idx), cfg.embed_dim, cfg.hidden_dim).to(device)

# Track gradients and parameter histograms automatically
wandb.watch(model, log="all", log_freq=50)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
wandb.config.update({"total_params": total_params, "trainable_params": trainable_params})
print("Total params:", total_params)

pos_weight = torch.tensor([cfg.pos_weight]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.num_epochs)

best_f1 = 0
global_step = 0

for epoch in range(cfg.num_epochs):

    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.float().to(device)

        optimizer.zero_grad()

        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()
        global_step += 1

        # per-step logging (lightweight, high-resolution loss curve)
        wandb.log({
            "train/step_loss": loss.item(),
            "train/grad_norm": float(grad_norm),
            "train/lr": optimizer.param_groups[0]["lr"],
            "global_step": global_step,
        })

    scheduler.step()

    avg_train_loss = total_loss / len(train_loader)

    model.eval()

    preds = []
    probs_all = []
    truths = []
    val_loss_total = 0

    with torch.no_grad():
        for batch_x, batch_y in val_loader:

            batch_x = batch_x.to(device)
            batch_y_dev = batch_y.float().to(device)

            logits = model(batch_x)
            val_loss_total += criterion(logits, batch_y_dev).item()

            probs = torch.sigmoid(logits)
            pred = (probs > 0.5).long()

            preds.extend(pred.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())
            truths.extend(batch_y.numpy())

    acc = accuracy_score(truths, preds)
    f1 = f1_score(truths, preds)
    precision = precision_score(truths, preds, zero_division=0)
    recall = recall_score(truths, preds, zero_division=0)
    avg_val_loss = val_loss_total / len(val_loader)

    print(
        f"Epoch {epoch+1}"
        f" | Acc={acc:.4f}"
        f" | F1={f1:.4f}"
    )

    # ---- epoch-level wandb logging ----
    wandb.log({
        "epoch": epoch + 1,
        "train/epoch_loss": avg_train_loss,
        "val/loss": avg_val_loss,
        "val/accuracy": acc,
        "val/f1": f1,
        "val/precision": precision,
        "val/recall": recall,
    })

    # ROC and PR curves (per epoch, so you can watch them evolve)
    wandb.log({
        "val/roc_curve": wandb.plot.roc_curve(
            np.array(truths), np.column_stack([1 - np.array(probs_all), np.array(probs_all)]),
            labels=["not_answer", "answer"]
        ),
        "val/pr_curve": wandb.plot.pr_curve(
            np.array(truths), np.column_stack([1 - np.array(probs_all), np.array(probs_all)]),
            labels=["not_answer", "answer"]
        ),
    })

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "/kaggle/working/best_lstm.pth")
        wandb.run.summary["best_f1"] = best_f1
        wandb.run.summary["best_epoch"] = epoch + 1

print("\nBest F1:", best_f1)
wandb.run.summary["final_best_f1"] = best_f1


Total params: 3607041
Epoch 1 | Acc=0.8010 | F1=0.1310
Epoch 2 | Acc=0.7970 | F1=0.2509
Epoch 3 | Acc=0.2370 | F1=0.3428
Epoch 4 | Acc=0.3930 | F1=0.3263
Epoch 5 | Acc=0.6860 | F1=0.4353
Epoch 6 | Acc=0.7360 | F1=0.5672
Epoch 7 | Acc=0.8100 | F1=0.6643
Epoch 8 | Acc=0.8520 | F1=0.7259
Epoch 9 | Acc=0.8550 | F1=0.7330
Epoch 10 | Acc=0.8540 | F1=0.7326

Best F1: 0.7329650092081031


## Evaluation (last epoch's validation predictions)

In [14]:
print("\nClassification Report")
report_str = classification_report(truths, preds)
print(report_str)

report_dict = classification_report(truths, preds, output_dict=True)
report_table = wandb.Table(
    columns=["class", "precision", "recall", "f1-score", "support"],
    data=[
        [k, v["precision"], v["recall"], v["f1-score"], v["support"]]
        for k, v in report_dict.items()
        if k in ("0", "1")
    ]
)
wandb.log({"eval/classification_report": report_table})
wandb.log({"eval/final_accuracy": report_dict["accuracy"]})

print("\nConfusion Matrix")
cm = confusion_matrix(truths, preds)
print(cm)

wandb.log({
    "eval/confusion_matrix": wandb.plot.confusion_matrix(
        y_true=np.array(truths).astype(int),
        preds=np.array(preds).astype(int),
        class_names=["not_answer", "answer"]
    )
})



Classification Report
              precision    recall  f1-score   support

           0       1.00      0.82      0.90       800
           1       0.58      1.00      0.73       200

    accuracy                           0.85      1000
   macro avg       0.79      0.91      0.82      1000
weighted avg       0.92      0.85      0.87      1000


Confusion Matrix
[[654 146]
 [  0 200]]


## Test Predictions

In [15]:
test_dataset = MCQDataset(test, vocab, cfg.max_len)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

model.load_state_dict(torch.load("/kaggle/working/best_lstm.pth"))
model.eval()

all_scores = []

with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        probs = torch.sigmoid(logits)
        all_scores.extend(probs.cpu().numpy())


## Top-3 Options

In [16]:
predictions = []

for i in range(len(test)):
    scores = all_scores[i * 5:(i + 1) * 5]
    top3 = np.argsort(scores)[::-1][:3]
    pred = " ".join(["ABCDE"[x] for x in top3])
    predictions.append(pred)

# Log distribution of the model's top-1 pick across A-E on the test set (sanity check for bias)
top1_letters = [p.split(" ")[0] for p in predictions]
top1_counts = pd.Series(top1_letters).value_counts().sort_index()
wandb.log({
    "test/top1_distribution": wandb.plot.bar(
        wandb.Table(data=[[k, v] for k, v in top1_counts.items()], columns=["letter", "count"]),
        "letter", "count",
        title="Test set: predicted top-1 answer letter distribution"
    )
})


## Submission

In [17]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

# Log the submission file itself as a wandb artifact so it's versioned with the run
sub_artifact = wandb.Artifact("mcq-submission", type="predictions")
sub_artifact.add_file("/kaggle/working/submission.csv")
wandb.log_artifact(sub_artifact)

# Log the trained model weights as a model artifact
model_artifact = wandb.Artifact("lstm-mcq-model", type="model")
model_artifact.add_file("/kaggle/working/best_lstm.pth")
wandb.log_artifact(model_artifact)

wandb.finish()

print("\nDone. Submission saved to /kaggle/working/submission.csv")


wandb: uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA; uploading artifact run-nsrtdk6p-valpr_curve_table-TP-fJA; uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA; uploading artifact run-nsrtdk6p-valpr_curve_table-TP-fJA; uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA (+ 19 more)
wandb: uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA; uploading artifact run-nsrtdk6p-valpr_curve_table-TP-fJA; uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA; uploading artifact run-nsrtdk6p-valpr_curve_table-TP-fJA; uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA (+ 20 more)
wandb: uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA; uploading artifact run-nsrtdk6p-valpr_curve_table-TP-fJA; uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA; uploading artifact run-nsrtdk6p-valpr_curve_table-TP-fJA; uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA (+ 19 more)
wandb: uploading artifact run-nsrtdk6p-valroc_curve_table-LW3vYA; u


Done. Submission saved to /kaggle/working/submission.csv
